# Day 04：自回归循环的浪费

对应 [`docs/day03-04.md`](../docs/day03-04.md) 的**任务 8**，是 Day 4 最重要的产出。

目的是把「无 KV Cache 有多浪费」**测出来**，为学习单元 Day 26～30 实现 Cache 留一个 baseline。

注意今天这个模型**没有 Attention**，所以重复计算的只是 embedding 查表和一个 GEMM。
等 Attention 加上之后浪费会更严重，因为 Attention 的中间矩阵随 `S²` 增长。

## 1. 输入长度单调递增

在 `generate_greedy` 的循环里打印每轮的 `input_ids.shape[1]`：

```text
第 1 轮 forward 10 个 token → 产出第 11 个
第 2 轮 forward 11 个 token → 产出第 12 个   ← 前 10 个又算了一遍
第 3 轮 forward 12 个 token → 产出第 13 个   ← 前 11 个又算了一遍
```

In [ ]:
# 跑一次生成，打印每轮的 S


## 2. 重算的结果每轮完全相同

这是「浪费」成立的前提：如果重算的结果不同，那就不算浪费。

把第 1 轮和第 2 轮**前 10 个位置**的 hidden_states 抓出来对比，
应当逐元素相等（用 `torch.testing.assert_close`）。

In [ ]:
# 对比相邻两轮前缀位置的 hidden_states


## 3. 累计计算量对比

累加总共处理了多少个 token-位置，和「理想情况」（有 Cache）对比。

公式：无 Cache 是 `Σ(P .. P+N-1)`，有 Cache 是 `P + (N-1)`。

**自己动手算这两行，别只是抄表**：

| prompt 长度 | 生成数 | 无 Cache 累计 | 有 Cache 累计 | 倍数 |
|---|---|---|---|---|
| 10 | 20 | 390 | 29 | 13.4× |
| 1024 | 100 | 107,350 | 1,123 | 95.6× |

In [ ]:
# 用公式算累计量，验证上表两行，并扫几组 (P, N) 看倍数怎么涨


## 4. 结论

把结论写进 `docs/concepts/03-autoregressive-loop.md`（任务 6 新建）。

一句话版本：**这个浪费就是 KV Cache 存在的全部理由。**